In [1]:
import pandas as pd
import numpy as np
import json
from IPython.display import display

#  Load dataset (TA nulls kept as NaN) 
df = pd.read_csv("dataset.csv")

# Detect sessions with no TA annotation
Missing_TA = (
    df.groupby("session")["TA"]
    .apply(lambda x: x.isnull().all())
)
Missing_TA = Missing_TA[Missing_TA].index.tolist()

# Load the 3 test episodes selected in Step 1
with open("test_episodes.json") as f:
    test_episodes = json.load(f)

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Sessions with no TA annotation: {Missing_TA}")
print(f"Test episodes loaded: {list(test_episodes.keys())}")


Dataset loaded: 10,799 rows x 13 columns
Sessions with no TA annotation: [2, 3, 7, 19, 24, 26, 28]
Test episodes loaded: ['High', 'Low', 'Mixed']


In [2]:
#time conversion 

def to_seconds(t):
    """Convert HH:MM:SS string to total seconds."""
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

df["start_sec"]= df["start"].apply(to_seconds)
df["end_sec"]= df["end"].apply(to_seconds)
df["duration"]= df["end_sec"] - df["start_sec"]# utterance duration in seconds
df["word_count"]= df["content"].str.split().str.len() # words per utterance

print("Time columns added: start_sec, end_sec, duration, word_count")
display(df[["speaker", "start", "end", "start_sec", "end_sec", "duration", "word_count"]].head(5))

Time columns added: start_sec, end_sec, duration, word_count


,speaker,start,end,start_sec,end_sec,duration,word_count
0,Christopher,00:00:02,00:00:05,2,5,3,6
1,Brian,00:00:10,00:02:04,10,124,114,273
2,Brian,00:02:08,00:03:03,128,183,55,87
3,Christopher,00:03:03,00:03:05,183,185,2,4
4,Brian,00:03:05,00:04:11,185,251,66,146


In [ ]:
#helper function to note the distribution of time spent speakign accross participants 
def gini(values):
    """
    Gini coefficient for participation equity.
    0 = perfectly equal speaking time across all participants.
    1 = one person does all the talking.
    """
    arr = np.array(sorted(values), dtype=float)
    n = len(arr)
    if n == 0 or arr.sum() == 0:
        return 0.0
    index = np.arange(1, n + 1)
    return (2 * (index * arr).sum()) / (n * arr.sum()) - (n + 1) / n


In [ ]:
#feature extraction function 
def extract_features(ep_df):
    """
    Extract temporal, participation, and conversational features
    from a single episode's utterance DataFrame.

    Parameters:
    ep_df : pd.DataFrame
    Subset of the main dataframe for one episode.

    Returns:
    dict of features
    """
    ep_df = ep_df.copy().sort_values("start_sec").reset_index(drop=True)

    # Temporal 
    total_duration = ep_df["end_sec"].max() - ep_df["start_sec"].min()
    n_turns = len(ep_df)
    avg_turn_length = ep_df["duration"].mean()

    # Response latency: the gap between end of one utterance and start of the next
    gaps = ep_df["start_sec"].iloc[1:].values - ep_df["end_sec"].iloc[:-1].values
    avg_response_lat = float(gaps.mean()) if len(gaps) > 0 else 0.0

    # Participation
    speak_time = ep_df.groupby("speaker")["duration"].sum()
    dominant_speaker = speak_time.idxmax()
    gini_score = gini(speak_time.values)

    #  Conversational 
    question_count = int(ep_df["content"].str.contains(r"\?").sum())
    question_density = question_count / n_turns
    avg_utt_length = ep_df["word_count"].mean()

    return {
        # temporal
        "total_duration_sec": round(total_duration, 2),
        "n_turns": n_turns,
        "avg_turn_length_sec": round(avg_turn_length, 2),
        "avg_response_latency": round(avg_response_lat, 2),
        # participation
        "n_speakers": len(speak_time),
        "dominant_speaker": dominant_speaker,
        "gini_coefficient": round(gini_score, 3),
        "speaking_time_sec": speak_time.to_dict(),
        # conversational
        "question_count": question_count,
        "question_density" : round(question_density, 3),
        "avg_utt_length_words": round(avg_utt_length, 2),
    }




In [ ]:
#test on example episodes 
results = {}

for label, meta in [(k, v["meta"]) for k, v in test_episodes.items()]:
    s, e = meta["session"], meta["ep"]
    ep_data = df[(df["session"] == s) & (df["ep"] == e)]
    feats = extract_features(ep_data)
    results[label] = feats

    print(f"\n{'-'*55}")
    print(f"  [{label} regulation]  Session={s}, Episode={e}")
    print(f"{'-'*55}")
    print(f" TEMPORAL:")
    print(f"  Total duration - {feats['total_duration_sec']}s")
    print(f"  Turns - {feats['n_turns']}")
    print(f"  Avg turn length - {feats['avg_turn_length_sec']}s")
    print(f"  Avg response lat. - {feats['avg_response_latency']}s")
    print(f" PARTICIPATION:")
    print(f"  Speakers - {feats['n_speakers']}")
    print(f"  Dominant speaker - {feats['dominant_speaker']}")
    print(f"  Gini coefficient - {feats['gini_coefficient']}  (0=equal, 1=dominant)")
    print(f"  Speaking time (s) - {feats['speaking_time_sec']}")
    print(f" CONVERSATIONAL:")
    print(f"  Questions - {feats['question_count']}")
    print(f"  Question density - {feats['question_density']}")
    print(f"  Avg utt. length - {feats['avg_utt_length_words']} words")
print(f"\n{'-'*55}")


-------------------------------------------------------
  [High regulation]  Session=14, Episode=18
-------------------------------------------------------
  TEMPORAL:
    Total duration - 78s
    Turns - 12
    Avg turn length - 4.0s
    Avg response lat. - 2.73s
  PARTICIPATION:
    Speakers - 2
    Dominant speaker - Daniel
    Gini coefficient - 0.146  (0=equal, 1=dominant)
    Speaking time (s) - {'Daniel': 31, 'Susan': 17}
  CONVERSATIONAL:
    Questions - 3
    Question density - 0.25
    Avg utt. length - 7.83 words

-------------------------------------------------------
  [Low regulation]  Session=17, Episode=38
-------------------------------------------------------
  TEMPORAL:
    Total duration - 162s
    Turns - 16
    Avg turn length - 9.62s
    Avg response lat. - 0.53s
  PARTICIPATION:
    Speakers - 4
    Dominant speaker - James
    Gini coefficient - 0.276  (0=equal, 1=dominant)
    Speaking time (s) - {'Anthony': 34, 'Daniel': 26, 'James': 74, 'Susan': 20}
  CONVE

In [9]:
#summary comparison tabel 

summary_rows = []
for label, feats in results.items():
    summary_rows.append({
        "Episode type" : label,
        "Duration (s)" : feats["total_duration_sec"],
        "Turns" : feats["n_turns"],
        "Avg turn (s)" : feats["avg_turn_length_sec"],
        "Avg latency (s)" : feats["avg_response_latency"],
        "Speakers" : feats["n_speakers"],
        "Gini" : feats["gini_coefficient"],
        "Questions" : feats["question_count"],
        "Question density" : feats["question_density"],
        "Avg utt. (words)" : feats["avg_utt_length_words"],
    })

summary_df = pd.DataFrame(summary_rows).set_index("Episode type")
display(summary_df)

,Duration (s),Turns,Avg turn (s),Avg latency (s),Speakers,Gini,Questions,Question density,Avg utt. (words)
Episode type,,,,,,,,,
High,78,12,4.00,2.73,2,0.146,3,0.250,7.83
Low,162,16,9.62,0.53,4,0.276,5,0.312,21.69
Mixed,147,13,8.62,2.92,4,0.375,6,0.462,20.62


In [13]:
#helper function to get the dominant feature 

def dominant_type(counts_dict):
    """
    Return the key with the highest count.
    Returns 'none' if all values are zero (no challenge/regulation present).
    """
    if sum(counts_dict.values()) == 0:
        return "none"
    return max(counts_dict, key=counts_dict.get)



In [ ]:
def extract_all_features(ep_df, session_id):
    """
    Extract all features for a single episode — temporal, participation,
    conversational plus challenge and regulatory .

    Parameters:
    ep_df : pd.DataFrame (rows for one episode)
    session_id : int  (used to handle TA annotation gaps)

    Returns:
    dict of features (one row in the final CSV)
    """
    ep_df = ep_df.copy().sort_values("start_sec").reset_index(drop=True)
    n = len(ep_df)

    # Temporal 
    total_duration = ep_df["end_sec"].max() - ep_df["start_sec"].min()
    avg_turn_length = ep_df["duration"].mean()
    gaps = ep_df["start_sec"].iloc[1:].values - ep_df["end_sec"].iloc[:-1].values
    avg_latency  = float(gaps.mean()) if len(gaps) > 0 else 0.0

    # Participation 
    speak_time= ep_df.groupby("speaker")["duration"].sum()
    gini_score= gini(speak_time.values)
    n_speakers = len(speak_time)

    # Conversational 
    question_count = int(ep_df["content"].str.contains(r"\?").sum())
    question_density = question_count / n
    avg_utt_length= ep_df["word_count"].mean()

    #  Challenge features 
    # challenge_rate: proportion of utterances with ANY challenge label = 1
    challenge_cols= {"C": ep_df["C"].sum(), "E": ep_df["E"].sum(), "M": ep_df["M"].sum(), "T": ep_df["T"].sum()}
    challenge_rate = ep_df[["C", "E", "M", "T"]].any(axis=1).mean()
    dominant_challenge= dominant_type(challenge_cols)

    # Regulation features 
    # TA excluded from unannotated sessions — kept as NaN rather than filled with 0
    ta_unknown = session_id in Missing_TA

    if ta_unknown:
        reg_cols = {"MC": ep_df["MC"].sum(), "RA": ep_df["RA"].sum()}
        regulation_rate = ep_df[["MC", "RA"]].any(axis=1).mean()
        ta_rate = np.nan
    else:
        reg_cols= {"MC": ep_df["MC"].sum(), "TA": ep_df["TA"].fillna(0).sum(), "RA": ep_df["RA"].sum()}
        regulation_rate = ep_df[["MC", "TA", "RA"]].any(axis=1).mean()
        ta_rate= ep_df["TA"].fillna(0).mean()

    dominant_regulation = dominant_type(reg_cols)

    return {
        # identifiers
        "session": session_id,
        "ep": int(ep_df["ep"].iloc[0]),
        # temporal
        "total_duration_sec":round(total_duration, 2),
        "n_turns": n,
        "avg_turn_length_sec": round(avg_turn_length, 2),
        "avg_response_latency": round(avg_latency, 2),
        # participation
        "n_speakers": n_speakers,
        "gini_coefficient": round(gini_score, 3),
        # conversational
        "question_count": question_count,
        "question_density": round(question_density, 3),
        "avg_utt_length_words": round(avg_utt_length, 2),
        # challenge 
        "challenge_rate": round(challenge_rate, 3),
        "C_rate":round(ep_df["C"].mean(), 3),
        "E_rate":round(ep_df["E"].mean(), 3),
        "M_rate":round(ep_df["M"].mean(), 3),
        "T_rate":round(ep_df["T"].mean(), 3),
        "dominant_challenge": dominant_challenge,
        # regulation 
        "regulation_rate": round(regulation_rate, 3),
        "MC_rate":round(ep_df["MC"].mean(), 3),
        "TA_rate":round(ta_rate, 3) if not np.isnan(ta_rate) else np.nan,
        "RA_rate":round(ep_df["RA"].mean(), 3),
        "dominant_regulation": dominant_regulation,
        "ta_annotated": not ta_unknown,
    }



In [17]:
#test on example episodes 

results = {}

for label, meta in [(k, v["meta"]) for k, v in test_episodes.items()]:
    s, e = meta["session"], meta["ep"]
    ep_data = df[(df["session"] == s) & (df["ep"] == e)]
    feats= extract_all_features(ep_data, s)
    results[label] = feats

    print(f"\n{'-'*55}")
    print(f"  [{label} regulation]  Session={s}, Episode={e}")
    print(f"{'-'*55}")
    print(f"  TEMPORAL:")
    print(f"    Total duration - {feats['total_duration_sec']}s")
    print(f"    Turns - {feats['n_turns']}")
    print(f"    Avg turn length - {feats['avg_turn_length_sec']}s")
    print(f"    Avg response lat. - {feats['avg_response_latency']}s")
    print(f"  PARTICIPATION:")
    print(f"    Speakers - {feats['n_speakers']}")
    print(f"    Gini coefficient - {feats['gini_coefficient']}")
    print(f"  CONVERSATIONAL: ")
    print(f"    Questions - {feats['question_count']}  (density={feats['question_density']})")
    print(f"    Avg utt. length -  {feats['avg_utt_length_words']} words")
    print(f"  CHALLENGE:")
    print(f"    Challenge rate - {feats['challenge_rate']}  (dominant={feats['dominant_challenge']})")
    print(f"    C={feats['C_rate']}  E={feats['E_rate']}  M={feats['M_rate']}  T={feats['T_rate']}")
    print(f"  REGULATIO:")
    print(f"    Regulation rate -  {feats['regulation_rate']}  (dominant={feats['dominant_regulation']})")
    print(f"    MC={feats['MC_rate']}  TA={feats['TA_rate']}  RA={feats['RA_rate']}")
    print(f"    TA annotated  -  {feats['ta_annotated']}")
print(f"\n{'-'*55}")



-------------------------------------------------------
  [High regulation]  Session=14, Episode=18
-------------------------------------------------------
  TEMPORAL:
    Total duration - 78s
    Turns - 12
    Avg turn length - 4.0s
    Avg response lat. - 2.73s
  PARTICIPATION:
    Speakers - 2
    Gini coefficient - 0.146
  CONVERSATIONAL: 
    Questions - 3  (density=0.25)
    Avg utt. length -  7.83 words
  CHALLENGE:
    Challenge rate - 0.083  (dominant=T)
    C=0.0  E=0.0  M=0.0  T=0.083
  REGULATIO:
    Regulation rate -  1.0  (dominant=MC)
    MC=1.0  TA=0.0  RA=0.0
    TA annotated  -  True

-------------------------------------------------------
  [Low regulation]  Session=17, Episode=38
-------------------------------------------------------
  TEMPORAL:
    Total duration - 162s
    Turns - 16
    Avg turn length - 9.62s
    Avg response lat. - 0.53s
  PARTICIPATION:
    Speakers - 4
    Gini coefficient - 0.276
  CONVERSATIONAL: 
    Questions - 5  (density=0.312)
    A

In [18]:
summary_rows = []
for label, feats in results.items():
    summary_rows.append({
        "Episode"  : label,
        "Duration (s)" : feats["total_duration_sec"],
        "Turns" : feats["n_turns"],
        "Gini" : feats["gini_coefficient"],
        "Challenge rate": feats["challenge_rate"],
        "Dom. challenge" : feats["dominant_challenge"],
        "Regulation rate" : feats["regulation_rate"],
        "Dom. regulation" : feats["dominant_regulation"],
        "TA annotated": feats["ta_annotated"],
    })

summary_df = pd.DataFrame(summary_rows).set_index("Episode")
display(summary_df)

,Duration (s),Turns,Gini,Challenge rate,Dom. challenge,Regulation rate,Dom. regulation,TA annotated
Episode,,,,,,,,
High,78,12,0.146,0.083,T,1.000,MC,True
Low,162,16,0.276,0.062,T,0.000,none,True
Mixed,147,13,0.375,0.231,M,0.769,MC,True


In [ ]:
all_features = []

for (session, ep), group in df.groupby(["session", "ep"]):
    feats = extract_all_features(group, session)
    all_features.append(feats)

features_df = pd.DataFrame(all_features)

print(f"Feature dataset: {features_df.shape[0]} episodes × {features_df.shape[1]} features")

# report the features accross all episodes 
print("\n" + "-" * 55)
print(" Feature Report ")
print("-" * 55)

print(f"\n1. Sessions covered : {features_df['session'].nunique()} / 28")
ep_counts = features_df.groupby("session").size()
print(f"  Episodes per session — min: {ep_counts.min()}, max: {ep_counts.max()}")

print(f"\n2. Nulls")
nulls = features_df.isnull().sum()
nulls = nulls[nulls > 0]
print(nulls.to_string() if len(nulls) > 0 else "   None ")
print(f"   (TA_rate nulls expected for sessions {Missing_TA})")

print(f"\n3. Range checks (all rates must be 0-1)")
rate_cols = ["challenge_rate", "C_rate", "E_rate", "M_rate", "T_rate",
             "regulation_rate", "MC_rate", "TA_rate", "RA_rate",
             "gini_coefficient", "question_density"]
all_ok = True
for col in rate_cols:
    mn, mx = features_df[col].min(), features_df[col].max()
    flag = " OUT OF RANGE" if mn < 0 or mx > 1 else ""
    if flag: all_ok = False
    print(f"   {col:<25} min={mn:.3f}  max={mx:.3f}{flag}")
if all_ok:
    print(" All within the range")

print(f"\n4. Dominant type distributions")
print(f"   dominant_challenge:\n{features_df['dominant_challenge'].value_counts().to_string()}")
print(f"\n   dominant_regulation:\n{features_df['dominant_regulation'].value_counts().to_string()}")

print(f"\n5. TA annotation flag")
print(f"   ta_annotated=True  : {features_df['ta_annotated'].sum()} episodes")
print(f"   ta_annotated=False : {(~features_df['ta_annotated']).sum()} episodes")

print(f"\n6. Key stats")
display(features_df[["n_turns", "total_duration_sec", "gini_coefficient",
                      "challenge_rate", "regulation_rate"]].describe().round(3))



Feature dataset: 882 episodes × 23 features

-------------------------------------------------------
 Feature Report 
-------------------------------------------------------

1. Sessions covered : 28 / 28
  Episodes per session — min: 12, max: 56

2. Nulls
TA_rate    242
   (TA_rate nulls expected for sessions [2, 3, 7, 19, 24, 26, 28])

3. Range checks (all rates must be 0-1)
   challenge_rate            min=0.000  max=1.000
   C_rate                    min=0.000  max=1.000
   E_rate                    min=0.000  max=1.000
   M_rate                    min=0.000  max=1.000
   T_rate                    min=0.000  max=1.000
   regulation_rate           min=0.000  max=1.000
   MC_rate                   min=0.000  max=1.000
   TA_rate                   min=0.000  max=1.000
   RA_rate                   min=0.000  max=1.000
   gini_coefficient          min=0.000  max=0.699
   question_density          min=0.000  max=1.000
   All within the range

4. Dominant type distributions
   dominant_ch

,n_turns,total_duration_sec,gini_coefficient,challenge_rate,regulation_rate
count,882.000,882.000,882.000,882.000,882.000
mean,12.244,90.291,0.285,0.180,0.559
std,14.529,162.229,0.168,0.205,0.340
min,1.000,1.000,0.000,0.000,0.000
25%,4.000,17.000,0.167,0.050,0.278
50%,8.000,47.500,0.300,0.125,0.667
75%,15.000,108.000,0.401,0.250,0.857
max,130.000,3608.000,0.699,1.000,1.000


In [ ]:
#save extracted features
features_df.to_csv("episode_features.csv", index=False)

print(f"Saved: episode_features.csv")
print(f"  {len(features_df)} episodes x {len(features_df.columns)} features")



Saved: episode_features.csv
  882 episodes x 23 features
